# DataType - Rust

All 13 Rust examples from [docs/datatype.md](https://platob.github.io/yggdryl/datatype/), in page order.

Generated by `scripts/build_docs_notebooks.py` from the blocks that
`scripts/check_docs_examples.py` compiles and runs, so every code cell below is
an example that passed. An edit here lives until the next build overwrites it.

The cells are unexecuted and expect the
[evcxr](https://github.com/evcxr/evcxr) kernel. Declare the crate once, in
a cell of your own, before running them:

```rust
:dep yggdryl = { version = "0.1", features = ["parquet", "iceberg"] }
```

In [ ]:
use yggdryl::DataType;

let value = DataType::from_str("decimal(18, 4)")?;
assert_eq!(value, DataType::decimal128(18, 4)?);

// Display is canonical and both text forms round-trip.
assert_eq!(value.to_string(), "decimal128(18,4)");
assert_eq!(DataType::from_str(&value.to_string())?, value);
assert_eq!(DataType::from_json(&value.clone().into_json()?)?, value);

## Children

In [ ]:
use yggdryl::{DataType, Field};

let quote = DataType::from_fields([
    Field::new("symbol", DataType::Utf8, false),
    Field::new("levels", DataType::list(DataType::Float64.nullable_field("item")), true),
])?;

assert_eq!(quote.field_len(), 2);
assert_eq!(quote.get_field(0).map(Field::name), Some("symbol"));
assert_eq!(quote.get_field_by_name("levels").unwrap().data_type().field_len(), 1);
assert!(quote.get_field_by_name("missing").is_none());

// Every child-bearing type answers the same two questions.
let lookup = DataType::map_of(DataType::Utf8, DataType::Int64, true)?;
assert_eq!(lookup.field_len(), 1);
assert_eq!(lookup.get_field(0).map(Field::name), Some("entries"));
assert!(lookup.as_fields().is_none() && quote.as_fields().is_some());

## Precision and resolution pick the width

In [ ]:
use yggdryl::{DataType, TimeUnit};

assert_eq!(DataType::decimal(38, 4)?, DataType::decimal128(38, 4)?);
assert_eq!(DataType::decimal(39, 4)?, DataType::decimal256(39, 4)?);
assert_eq!(DataType::time(TimeUnit::Second)?, DataType::Time32(TimeUnit::Second));
assert_eq!(DataType::time(TimeUnit::Nanosecond)?, DataType::Time64(TimeUnit::Nanosecond));

// Out-of-range parameters and mismatched unit categories are refused.
assert!(DataType::decimal(2, 3).is_err());
assert!(DataType::decimal128(39, 0).is_err());
assert!(DataType::time32(TimeUnit::Nanosecond).is_err());
assert!(DataType::time(TimeUnit::YearMonth).is_err());
assert!(DataType::fixed_size_binary(-1).is_err());

## Encodings that wrap a value

In [ ]:
use yggdryl::{DataType, DataTypeKind, Field};

let codes = DataType::dictionary(DataType::Int16, DataType::Utf8)?;
let runs = DataType::run_end_encoded(
    Field::new("run_ends", DataType::Int32, false),
    Field::new("values", DataType::Utf8, true),
)?;

assert_eq!(codes.kind(), DataTypeKind::Dictionary);
assert_eq!(runs.kind(), DataTypeKind::RunEndEncoded);
// A wrapper reports the shape of what it encodes, not of its own storage.
assert!(!codes.is_nested() && !runs.is_nested());

let DataType::Dictionary(dictionary) = &codes else { panic!("dictionary") };
assert_eq!(dictionary.key(), &DataType::Int16);
assert_eq!(dictionary.value(), &DataType::Utf8);

// The key must be an integer; run ends must be a non-null int16, int32, or int64.
assert!(DataType::dictionary(DataType::Utf8, DataType::Utf8).is_err());
assert!(DataType::run_end_encoded(
    Field::new("run_ends", DataType::UInt32, false),
    Field::new("values", DataType::Utf8, true),
).is_err());

## Unions and the dense-union sugar

In [ ]:
use yggdryl::{DataType, Field, UnionMode};

let members = [
    Field::new("number", DataType::Int64, false),
    Field::new("text", DataType::Utf8, true),
];
let members_union = DataType::dense_union(members.clone())?;

// The member sugar is not a second logical type: it is the dense union
// with IDs 0.. - and bare `variant` is a datatype of its own, so the
// parenthesis is what disambiguates the two spellings.
assert_eq!(
    members_union,
    DataType::union(
        [(0, members[0].clone()), (1, members[1].clone())],
        UnionMode::Dense,
    )?
);
assert_eq!(members_union.name(), "union");
assert!(members_union.to_string().starts_with("union(dense,"));
assert_eq!(DataType::from_str("variant(number:int64,text:string)")?.name(), "union");
assert_eq!(DataType::from_str("variant")?, DataType::Variant);

// An explicit union picks its own mode and its own non-negative type IDs.
let sparse = DataType::union(
    [(7, Field::new("only", DataType::Int32, false))],
    UnionMode::Sparse,
)?;
assert_eq!(sparse.get_field(0).map(Field::name), Some("only"));
assert!(DataType::union(
    [(0, members[0].clone()), (0, members[1].clone())],
    UnionMode::Dense,
).is_err());

## Variant, geometry, and geography

In [ ]:
use yggdryl::{DataType, DataTypeKind, EdgeAlgorithm};

// Bare `variant` is the self-describing semi-structured datatype; the
// parenthesis selects the dense-union sugar instead.
let variant = DataType::variant();
assert_eq!(variant.to_string(), "variant");
assert_eq!(variant.kind(), DataTypeKind::Variant);
assert_eq!(DataType::from_str("variant")?, variant);
assert_eq!(DataType::from_str("variant(n:int64)")?.name(), "union");

// The bare geospatial spellings fill the defaults Parquet and Iceberg
// share - `OGC:CRS84`, and spherical edges for a geography - so a
// parameter appears exactly when it says something.
let geometry = DataType::geometry(None)?;
assert_eq!(geometry.to_string(), "geometry");
assert_eq!(geometry, DataType::geometry(Some("OGC:CRS84"))?);
assert_eq!(geometry.kind(), DataTypeKind::Geospatial);
assert_eq!(
    DataType::geometry(Some("EPSG:3857"))?.to_string(),
    "geometry(\"EPSG:3857\")"
);

let geography = DataType::geography(None, None)?;
assert_eq!(geography.to_string(), "geography");
assert_eq!(
    geography,
    DataType::geography(Some("OGC:CRS84"), Some(EdgeAlgorithm::Spherical))?
);

let vincenty = DataType::geography(None, Some(EdgeAlgorithm::Vincenty))?;
assert_eq!(vincenty.to_string(), "geography(\"OGC:CRS84\",\"vincenty\")");
assert_eq!(DataType::from_str("geography('OGC:CRS84', 'vincenty')")?, vincenty);

// A geometry has no edge algorithm, and an empty CRS names nothing.
assert!(DataType::from_str("geometry('OGC:CRS84', 'vincenty')").is_err());
assert!(DataType::geometry(Some("")).is_err());

## Identity and family

In [ ]:
use yggdryl::{DataType, DataTypeId, DataTypeKind};

let stamp = DataType::from_str("timestamp(ns, Europe/Paris)")?;
assert_eq!(stamp.id(), DataTypeId::Timestamp);
assert_eq!(stamp.kind(), DataTypeKind::Temporal);
assert_eq!(stamp.name(), "timestamp");

// The id drops parameters, so two resolutions share one identity ...
assert_eq!(DataType::from_str("timestamp(s)")?.id(), stamp.id());
// ... while the values themselves stay distinct.
assert_ne!(DataType::from_str("timestamp(s)")?, stamp);

assert_eq!(DataType::decimal(38, 4)?.id(), DataTypeId::Decimal128);
assert_eq!(DataType::decimal(38, 4)?.kind(), DataTypeKind::Decimal);

## Arrow projection

In [ ]:
use yggdryl::{DataType, TimeUnit};

let value = DataType::from_str("map<string,array<decimal(38,18)>>")?;
let arrow = value.clone().into_arrow()?;

assert_eq!(DataType::from_arrow(&arrow)?, value);
assert_eq!(value.clone().into_arrow()?, arrow);
assert_eq!(DataType::try_from(arrow)?, value);

// Projection re-checks parameters, so a directly built enum value cannot escape.
assert!(DataType::Time32(TimeUnit::Nanosecond).into_arrow().is_err());
assert!(DataType::Time32(TimeUnit::Nanosecond).into_arrow_ffi().is_err());

## Default values

In [ ]:
use yggdryl::{DataType, Field, Scalar};

let value = DataType::from_fields([
    Field::new("id", DataType::Int32, false),
    Field::new("note", DataType::Utf8, true),
])?;

// One positional slot per child, each honoring its own nullability.
assert_eq!(
    value.default_value()?.as_sequence().unwrap(),
    &[Scalar::I64(0), Scalar::Null]
);
assert!(value.is_default_value(&value.default_value()?)?);
assert_eq!(DataType::Utf8.default_value()?, Scalar::String("".into()));

// A default is bounded: a layout too large to materialize is an error, not a null.
assert!(DataType::FixedSizeBinary(64 * 1024 * 1024 + 1).default_value().is_err());

## Serializing a schema

In [ ]:
use yggdryl::DataType;
use yggdryl::generic::Scalar;

let data_type = DataType::decimal128(9, 2)?;

// One structural model, three formats over it.
assert_eq!(DataType::from_value(data_type.clone().into_value())?, data_type);
assert_eq!(DataType::from_json(&data_type.clone().into_json()?)?, data_type);
assert_eq!(DataType::from_yaml(&data_type.clone().into_yaml()?)?, data_type);
assert_eq!(DataType::from_toml(&data_type.clone().into_toml()?)?, data_type);

let shape = data_type.into_value();
assert_eq!(shape.get_key_str("type").and_then(Scalar::as_utf8), Some("decimal128"));

## A readable rendering

In [ ]:
use yggdryl::DataType;

let rows = DataType::list(
    DataType::from_fields([DataType::Utf8.nullable_field("venue")])?.nullable_field("item"),
);

// Compact still round-trips.
assert_eq!(DataType::from_str(&rows.to_string())?, rows);

// Readable is the alternate, or the named adapter - one implementation.
assert_eq!(format!("{rows:#}"), rows.pretty().to_string());
assert_eq!(
    format!("{rows:#}"),
    "list\n  item: struct[1], nullable\n    venue: utf8, nullable",
);

## Compatibility rewriting

In [ ]:
use yggdryl::{DataType, Field, Scheme, TimeUnit};

let source = DataType::from_fields([
    Field::new("small", DataType::UInt8, false),
    Field::new("wide", DataType::UInt64, true),
    Field::new(
        "text",
        DataType::large_list(DataType::Utf8View.nullable_field("item")),
        false,
    ),
])?;

let spark = source.clone().into_scheme_compat(&Scheme::SPARK)?;
let rewritten = spark.as_fields().unwrap();
assert_eq!(rewritten[0].data_type(), &DataType::Int16);
assert_eq!(rewritten[1].data_type(), &DataType::decimal128(20, 0)?);
assert_eq!(
    rewritten[2].data_type(),
    &DataType::list(DataType::Utf8.nullable_field("item"))
);

// Arrow is a validated clone; Polars keeps the unsigned integers Spark has to widen.
assert_eq!(source.clone().into_scheme_compat(&Scheme::ARROW)?, source);
assert_eq!(DataType::UInt32.into_scheme_compat(&Scheme::POLARS)?, DataType::UInt32);

// A rewrite that would reinterpret values is refused, and the path is named.
let error = DataType::from_fields([Field::new(
    "created",
    DataType::Timestamp(TimeUnit::Nanosecond, None),
    false,
)])?
.into_scheme_compat(&Scheme::SPARK)
.unwrap_err()
.to_string();
assert!(error.contains("created") && error.contains("got ns"));

## Building the enum directly

In [ ]:
use yggdryl::{DataType, Field, TimeUnit};

let broken = DataType::Time32(TimeUnit::Nanosecond);
assert!(broken.validate().is_err());
assert!(DataType::time32(TimeUnit::Nanosecond).is_err());

// A valid value validates without allocating, recursing through every child.
let value = DataType::list(Field::new(
    "item",
    DataType::Decimal128 { precision: 18, scale: 4 },
    true,
));
value.validate()?;
assert!(DataType::Decimal128 { precision: 0, scale: 0 }.validate().is_err());

assert_eq!(DataType::PARSE_RECURSION_LIMIT, 64);